# Soldani - Second task - Benchmark

In [60]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [61]:
import json
import pandas as pd

from openai import OpenAI
from pgmpy.estimators import BayesianEstimator
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect, spurious_effect,
    natural_direct_effect, natural_indirect_effect,
)

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="not-needed",
)

#client = OpenAI()
MODEL_NAME = "qwen2.5-7b-instruct"

In [62]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
}

In [63]:
import time

def run_fairmind(config: dict) -> tuple[dict, float]:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
        sorted_mediators=len(config["mediators"]) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(BayesianEstimator, {"prior_type": "BDeu"}),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    effects = {
        "TV": total_variation(bn, target, config["protected"], x0, x1),
        "TE": total_effect(bn, target, config["protected"], x0, x1),
        "SE": spurious_effect(bn, target, config["protected"], x0),
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, elapsed

ground_truth, fairmind_time = run_fairmind(CONFIG)
print(f"FairMind — elapsed time: {fairmind_time:.4f}s")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

2026-07-01 10:02:14.473 | DEBUG    | src.model:fit_discrete_bayesian_model:33 - Using estimator: <class 'pgmpy.estimators.BayesianEstimator.BayesianEstimator'> with parameters: {'prior_type': 'BDeu'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'S2_gender': 'C', 'hours-per-week': 'N', 'education': 'C', 'T_income': 'C'}
2026-07-01 10:02:14.522 | DEBUG    | src.effects:total_variation:248 - Computing total variation for target=('T_income', '>50K'), private_baseline=Female, private_mod=Male
2026-07-01 10:02:14.525 | DEBUG    | src.effects:spurious_effect:195 - Computing spurious effect for target=('T_income', '>50K'), private_value=Female


FairMind — elapsed time: 0.0082s
  TV: 0.194470
  TE: 0.183161
  SE: -0.007296
  DE: 0.137049
  IE: -0.046112


In [64]:
def build_gpt_prompt(config: dict, n_rows: int = 2000) -> str:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()
    sample = df.sample(n=min(n_rows, len(df)), random_state=42)
    csv_str = sample.to_csv(index=False)

    return f"""You are a causal fairness expert. Compute five causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

DATASET ({len(sample)} rows):
{csv_str}

VARIABLE ROLES:
- X (protected): "{config['protected']}", x0="{config['x0']}", x1="{config['x1']}"
- Y (target):    "{config['target_col']}", target state="{config['target_val']}"
- W (mediators): {config['mediators']}
- Z (confounders): {config['confounders']}

IDENTIFICATION FORMULAE (use these exactly):
- TV = P(Y=y | X=x1) - P(Y=y | X=x0)
- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)
- SE = TV - TE
- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)
- IE = sum_z,w P(Y=y|x0,w,z) * [P(w|x1,z) - P(w|x0,z)] * P(z)

Return ONLY a JSON object, no other text:
{{
  "TV": <float>,
  "TE": <float>,
  "SE": <float>,
  "DE": <float>,
  "IE": <float>
}}"""

prompt = build_gpt_prompt(CONFIG)
print(prompt[:600], "\n[...]")

You are a causal fairness expert. Compute five causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

DATASET (2000 rows):
S2_gender,hours-per-week,education,T_income
Female,40,HS-grad,<=50K
Male,40,HS-grad,<=50K
Female,40,Bachelors,>50K
Male,40,HS-grad,<=50K
Female,30,Bachelors,<=50K
Female,40,HS-grad,<=50K
Male,45,HS-grad,<=50K
Female,40,Bachelors,>50K
Male,50,HS-grad,<=50K
Female,40,Some-college,<=50K
Male,70,Some-college,<=50K
Male,30,Bachelors,<=50K
Male,40,Some-college,<=50K
Male,45,11th,<=50K
Male,40,Bachelors,<=50K
Male,60,Assoc-voc,<=50K
Fema 
[...]


In [65]:
def call_gpt(prompt: str, model: str = MODEL_NAME) -> tuple[dict, dict, float]:
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    elapsed = time.perf_counter() - start

    usage = {
        "input_tokens":     response.usage.prompt_tokens,
        "output_tokens":    response.usage.completion_tokens,
        "reasoning_tokens": None,
        "total_tokens":     response.usage.total_tokens,
    }

    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json", "").replace("```", "").strip()
    effects = json.loads(raw)

    return effects, usage, elapsed

gpt_effects, gpt_usage, gpt_time = call_gpt(prompt)
print(f"LLM — time: {gpt_time:.4f}s")
print(f"Token: input={gpt_usage['input_tokens']}, "
      f"output={gpt_usage['output_tokens']}, "
      f"total={gpt_usage['total_tokens']}")
print(json.dumps(gpt_effects, indent=2))

INFO:openai._base_client:Retrying request to /chat/completions in 0.388202 seconds
INFO:httpx:HTTP Request: POST http://localhost:8080/v1/chat/completions "HTTP/1.1 200 OK"


LLM — time: 670.3290s
Token: input=27745, output=48, total=27793
{
  "TV": 0.0,
  "TE": 0.0,
  "SE": 0.0,
  "DE": 0.0,
  "IE": 0.0
}


In [66]:
def compute_discrepancies(ground_truth: dict, gpt_effects: dict) -> pd.DataFrame:
    rows = []
    for effect in ["TV", "TE", "SE", "DE", "IE"]:
        gt  = ground_truth.get(effect, float("nan"))
        gpt = float(gpt_effects.get(effect, float("nan")))
        abs_err = abs(gt - gpt)
        rel_err = abs_err / abs(gt) if abs(gt) > 1e-9 else float("nan")
        rows.append({
            "effect":      effect,
            "fairmind":    round(gt,  6),
            "gpt":         round(gpt, 6),
            "abs_error":   round(abs_err, 6),
            "rel_error_%": round(rel_err * 100, 2) if not pd.isna(rel_err) else float("nan"),
        })
    return pd.DataFrame(rows)

discrepancies = compute_discrepancies(ground_truth, gpt_effects)
print(discrepancies.to_string(index=False))

effect  fairmind  gpt  abs_error  rel_error_%
    TV  0.194470  0.0   0.194470        100.0
    TE  0.183161  0.0   0.183161        100.0
    SE -0.007296  0.0   0.007296        100.0
    DE  0.137049  0.0   0.137049        100.0
    IE -0.046112  0.0   0.046112        100.0


In [67]:
def save_results(config, ground_truth, gpt_effects, discrepancies, usage, fairmind_time, gpt_time):
    import os, datetime
    os.makedirs("benchmark_results", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"benchmark_results/{config['dataset_name']}_{ts}.json"

    out = {
        "dataset":       config["dataset_name"],
        "config":        {k: v for k, v in config.items() if k != "csv_path"},
        "fairmind":      ground_truth,
        "gpt":           gpt_effects,
        "discrepancies": discrepancies.to_dict(orient="records"),
        "token_usage":   usage,
        "timing": {
            "fairmind_seconds": round(fairmind_time, 4),
            "gpt_seconds":      round(gpt_time, 4),
        },
    }
    with open(fname, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved: {fname}")

save_results(CONFIG, ground_truth, gpt_effects, discrepancies, gpt_usage, fairmind_time, gpt_time)

Saved: benchmark_results/adult_20260701_101325.json


## Riassunto

Questo notebook esegue un benchmark che confronta FairMind (calcolo esatto via Bayesian Network) con un LLM (Qwen 2.5 7B) nel calcolo di 5 metriche di causal fairness (TV, TE, SE, DE, IE) sul dataset Adult.

### Cosa fa il notebook
1. Carica il dataset Adult processato e seleziona le colonne rilevanti (S2_gender, hours-per-week, education, T_income).
2. Costruisce un grafo SFM con `S2_gender` come attributo sensibile (X), `T_income` come outcome (Y), `hours-per-week` come mediatore (W), `education` come confondente (Z).
3. Fitta un Discrete Bayesian Network con pgmpy (BayesianEstimator, prior BDeu).
4. Calcola gli effetti causali esatti con FairMind (tempo: ~8ms).
5. Costruisce un prompt contenente 2000 righe campionate del dataset in formato CSV e lo invia all'LLM.
6. L'LLM produce una risposta JSON con i 5 effetti (tempo: ~670 secondi).
7. Confronta i risultati e calcola errore assoluto e relativo.
8. Salva i risultati su disco in formato JSON.

### Problemi emersi
- **LLM non in grado di calcolare effetti causali**: l'LLM ha restituito 0.0 per tutte le metriche, indicando che un modello linguistico (specialmente 7B parametri) non pu&ograve; eseguire inferenza probabilistica su Bayesian Network da dati tabellari grezzi.
- **Formula errata nel prompt**: l'identit&agrave; fornita `SE = TV - TE` &egrave; corretta solo se si considera `SE(x1) - SE(x0)`, non `SE(x0)` da sola. FairMind calcola `SE(x0)`, creando una discrepanza nella definizione.
- **Costo computazionale proibitivo**: 27.745 token di input e 670 secondi per una singola richiesta, insostenibile per un benchmark su larga scala.
- **Assenza di validazione**: l'output nullo (tutti zeri) non viene rilevato come fallimento, distorcendo le metriche di errore.

### Lezioni per iterazioni future
- Non chiedere all'LLM di calcolare effetti causali da dati grezzi; usare FairMind per il calcolo esatto e l'LLM solo per l'interpretazione dei risultati gi&agrave; calcolati.
- Inviare all'LLM i risultati precalcolati (JSON con le metriche) anzich&eacute; il dataset grezzo, riducendo drasticamente token e tempo.
- Correggere la formula di SE nel prompt se si vuole mantenere l'approccio "solo dataset".
- Aggiungere controlli sull'output LLM per rilevare fallimenti (es. valori nulli, NaN, output non JSON).
- Ridurre la dimensione del campione o usare statistiche riassuntive invece del CSV completo.